# Module 5.2 — Multiprocessing

### Python Mastery · Track 5: Concurrency & Asynchronous Programming

Threads cannot speed up heavy computation because of the Global Interpreter Lock. Multiprocessing sidesteps this by running work in **separate processes**, each with its own interpreter, giving true parallelism for CPU-bound tasks. This module covers processes, pools, sharing data, and the pitfalls unique to processes.

**A note on running this module.** Multiprocessing is unreliable when started directly inside a notebook cell, because worker processes cannot always import functions defined in the notebook, and the behaviour differs across operating systems. The reliable, portable approach, used throughout this notebook, is to write a small script to a `.py` file with the `%%writefile` command and run it with `!python`, so the code executes in a real script process exactly as it would in production. Each example therefore appears as a script you can also save and run yourself.

### Learning objectives

After completing this module you will be able to:

1. Run a function in a separate process with `multiprocessing.Process`.
2. Parallelise work across a `Pool` of worker processes.
3. Pass data between processes with `Queue` and return values from a pool.
4. Share state with `Value` and `Array`, and know why it needs care.
5. Avoid the common multiprocessing pitfalls, including the required `__main__` guard.

**Prerequisites:** Tracks 1 to 4, and Module 5.1.

---

## Part 1 · Why Processes, and the `__main__` Guard

A process is an independent program with its own memory and its own Python interpreter, so several processes genuinely run at the same time on multiple CPU cores. This makes multiprocessing the right tool for **CPU-bound** work: large computations, data crunching, image processing.

A firm requirement: code that starts processes must be protected by `if __name__ == "__main__":`. Without it, each new process re-imports the script and tries to start more processes, causing errors or an infinite cascade. Every example here follows that rule.

In [ ]:
%%writefile mp_basic.py
import multiprocessing as mp

def greet(name):
    # This runs in a separate process.
    print(f"hello from process for {name}")

if __name__ == "__main__":            # required guard for multiprocessing
    processes = [mp.Process(target=greet, args=(n,)) for n in ["A", "B", "C"]]
    for p in processes:
        p.start()                      # launch each process
    for p in processes:
        p.join()                       # wait for each to finish
    print("all processes done")

In [ ]:
# Run the script in a real process context, where multiprocessing works correctly.
!python mp_basic.py

## Part 2 · Process Pools for Parallel Work

Creating processes by hand is fine for a few tasks, but for applying the same function to many inputs, a `Pool` is far easier. It manages a fixed number of worker processes and distributes work among them. `pool.map` works like the built-in `map` but runs in parallel and returns results in input order.

In [ ]:
%%writefile mp_pool.py
import multiprocessing as mp

def heavy(n):
    # A stand-in for CPU-bound work: sum of squares up to n.
    return sum(i * i for i in range(n))

if __name__ == "__main__":
    inputs = [10_000, 20_000, 30_000, 40_000]
    with mp.Pool(processes=2) as pool:        # two worker processes
        results = pool.map(heavy, inputs)     # distribute the inputs, collect results in order
    for n, r in zip(inputs, results):
        print(f"heavy({n}) = {r}")

In [ ]:
!python mp_pool.py

The benefit appears with genuinely heavy work spread over multiple cores: the total time approaches the serial time divided by the number of workers (minus overhead). For light work, the cost of starting processes and moving data can outweigh the gain, so multiprocessing pays off mainly for substantial CPU-bound tasks.

## Part 3 · Passing Data Between Processes

Processes do not share memory by default, so data must be passed explicitly. A `multiprocessing.Queue` is a process-safe channel, much like the thread queue from Module 5.1 but designed to work across processes. Producers put items in; consumers take them out.

In [ ]:
%%writefile mp_queue.py
import multiprocessing as mp

def producer(q, count):
    for i in range(1, count + 1):
        q.put(i)
    q.put(None)                        # sentinel: signal the end

def consumer(q, out):
    while True:
        item = q.get()
        if item is None:
            break
        out.put(item * item)           # send results back via another queue
    out.put(None)

if __name__ == "__main__":
    work = mp.Queue()
    results = mp.Queue()
    p = mp.Process(target=producer, args=(work, 5))
    c = mp.Process(target=consumer, args=(work, results))
    p.start(); c.start()

    collected = []
    while True:
        item = results.get()
        if item is None:
            break
        collected.append(item)

    p.join(); c.join()
    print("squares received:", sorted(collected))

In [ ]:
!python mp_queue.py

## Part 4 · Sharing State with `Value` and `Array`

Sometimes processes need to share a piece of state directly. `multiprocessing.Value` and `Array` provide shared memory for a single value or an array of values. Because several processes may update them at once, you must guard updates with the lock that comes with them, just as with threads.

In [ ]:
%%writefile mp_shared.py
import multiprocessing as mp

def add_to_total(shared_total, amount, times):
    for _ in range(times):
        with shared_total.get_lock():        # the shared value carries its own lock
            shared_total.value += amount

if __name__ == "__main__":
    total = mp.Value("i", 0)                 # a shared integer ('i'), starting at 0
    procs = [
        mp.Process(target=add_to_total, args=(total, 1, 1000)),
        mp.Process(target=add_to_total, args=(total, 2, 1000)),
    ]
    for p in procs: p.start()
    for p in procs: p.join()
    print("shared total:", total.value)      # 1*1000 + 2*1000 = 3000, reliably

In [ ]:
!python mp_shared.py

## Part 5 · Pitfalls and Guidance

Multiprocessing has sharp edges worth remembering:

- Everything sent to a process (arguments, return values, queue items) must be **picklable**. Lambdas, local functions, and open file handles cannot be pickled, which causes errors.
- Start methods differ by platform: Linux defaults to `fork`, while macOS and Windows default to `spawn`, which re-imports your module, making the `__main__` guard essential.
- Starting processes and transferring data has real overhead; use multiprocessing for substantial CPU-bound work, not tiny tasks.
- Prefer a `Pool` or the `ProcessPoolExecutor` from Module 5.3 over manual process management for most work.

In [ ]:
%%writefile mp_guidance.py
import multiprocessing as mp

# A top-level (module-level) function IS picklable, so it works as a pool target.
# A lambda would NOT be picklable and would fail under the 'spawn' start method.
def cube(n):
    return n ** 3

if __name__ == "__main__":
    with mp.Pool(2) as pool:
        print("cubes:", pool.map(cube, range(1, 7)))

In [ ]:
!python mp_guidance.py

---

## Worked Examples

| Examples | Level |
|---|---|
| 1 and 2 | Easy |
| 3 and 4 | Medium |
| 5 and 6 | Difficult |

### Example 1 — A single worker process (Easy)

In [ ]:
%%writefile ex1.py
import multiprocessing as mp

def announce(message):
    print("worker says:", message)

if __name__ == "__main__":
    p = mp.Process(target=announce, args=("running in a separate process",))
    p.start()
    p.join()
    print("done")

In [ ]:
!python ex1.py

### Example 2 — A pool computing squares (Easy)

In [ ]:
%%writefile ex2.py
import multiprocessing as mp

def square(n):
    return n * n

if __name__ == "__main__":
    with mp.Pool(3) as pool:
        print(pool.map(square, [1, 2, 3, 4, 5]))

In [ ]:
!python ex2.py

### Example 3 — Parallel speed-up on heavy work (Medium)

In [ ]:
%%writefile ex3.py
import multiprocessing as mp
import time

def heavy(n):
    return sum(i * i for i in range(n))

if __name__ == "__main__":
    inputs = [2_000_000] * 4

    start = time.perf_counter()
    serial = [heavy(n) for n in inputs]          # one after another
    serial_time = time.perf_counter() - start

    start = time.perf_counter()
    with mp.Pool(2) as pool:                      # spread over workers
        parallel = pool.map(heavy, inputs)
    parallel_time = time.perf_counter() - start

    print(f"serial:   {serial_time:.2f}s")
    print(f"parallel: {parallel_time:.2f}s")
    print("results match:", serial == parallel)

In [ ]:
!python ex3.py

### Example 4 — Returning results through a queue (Medium)

In [ ]:
%%writefile ex4.py
import multiprocessing as mp

def worker(numbers, output):
    output.put(sum(numbers))            # put a partial sum into the shared queue

if __name__ == "__main__":
    output = mp.Queue()
    chunks = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
    procs = [mp.Process(target=worker, args=(chunk, output)) for chunk in chunks]
    for p in procs: p.start()

    partials = [output.get() for _ in chunks]    # collect one result per process
    for p in procs: p.join()
    print("partial sums:", sorted(partials), "| grand total:", sum(partials))

In [ ]:
!python ex4.py

### Example 5 — A shared counter with a lock (Difficult)

In [ ]:
%%writefile ex5.py
import multiprocessing as mp

def increment(counter, times):
    for _ in range(times):
        with counter.get_lock():
            counter.value += 1

if __name__ == "__main__":
    counter = mp.Value("i", 0)
    procs = [mp.Process(target=increment, args=(counter, 10_000)) for _ in range(4)]
    for p in procs: p.start()
    for p in procs: p.join()
    print("final count:", counter.value, "(expected 40000)")

In [ ]:
!python ex5.py

### Example 6 — A map-reduce style word count (Difficult)

In [ ]:
%%writefile ex6.py
import multiprocessing as mp
from collections import Counter

def count_words(text):
    return Counter(text.split())              # map step: count one chunk

if __name__ == "__main__":
    chunks = [
        "the cat sat",
        "the dog ran",
        "the cat ran the dog",
    ]
    with mp.Pool(3) as pool:
        partial_counts = pool.map(count_words, chunks)   # parallel map

    total = Counter()
    for c in partial_counts:                  # reduce step: merge the counts
        total += c
    print("word counts:", dict(sorted(total.items())))

In [ ]:
!python ex6.py

---

## Exercises

Write each solution into a `.py` file with `%%writefile`, then run it with `!python`, following the pattern above. Remember the `if __name__ == "__main__":` guard.

**Exercise 1 (Easy).** Write a script that starts two processes, each printing a different message, and waits for both.

In [ ]:
# Your %%writefile cell here


In [ ]:
# Your !python run cell here


**Exercise 2 (Easy).** Write a script using a `Pool` of 2 workers to compute the cube of each number from 1 to 6, printing the list.

In [ ]:
# Your %%writefile cell here


In [ ]:
# Your !python run cell here


**Exercise 3 (Medium).** Write a script where a pool applies a function `length(word)` to a list of words and prints the resulting list of lengths.

In [ ]:
# Your %%writefile cell here


In [ ]:
# Your !python run cell here


**Exercise 4 (Medium).** Write a script with a shared `Value` integer that two processes each increment 5,000 times under the value's lock; print the final total (expected 10,000).

In [ ]:
# Your %%writefile cell here


In [ ]:
# Your !python run cell here


**Exercise 5 (Difficult).** Write a script that splits the numbers 1 to 100 into four chunks, uses a pool to sum each chunk in parallel, and prints the partial sums and their total (which should be 5050).

In [ ]:
# Your %%writefile cell here


In [ ]:
# Your !python run cell here


---

## Solutions

**Solution 1**

In [ ]:
%%writefile sol1.py
import multiprocessing as mp

def show(msg):
    print(msg)

if __name__ == "__main__":
    a = mp.Process(target=show, args=("process one",))
    b = mp.Process(target=show, args=("process two",))
    a.start(); b.start()
    a.join(); b.join()

In [ ]:
!python sol1.py

**Solution 2**

In [ ]:
%%writefile sol2.py
import multiprocessing as mp

def cube(n):
    return n ** 3

if __name__ == "__main__":
    with mp.Pool(2) as pool:
        print(pool.map(cube, range(1, 7)))

In [ ]:
!python sol2.py

**Solution 3**

In [ ]:
%%writefile sol3.py
import multiprocessing as mp

def length(word):
    return len(word)

if __name__ == "__main__":
    words = ["apple", "fig", "banana", "kiwi"]
    with mp.Pool(2) as pool:
        print(pool.map(length, words))

In [ ]:
!python sol3.py

**Solution 4**

In [ ]:
%%writefile sol4.py
import multiprocessing as mp

def add(counter, times):
    for _ in range(times):
        with counter.get_lock():
            counter.value += 1

if __name__ == "__main__":
    counter = mp.Value("i", 0)
    procs = [mp.Process(target=add, args=(counter, 5000)) for _ in range(2)]
    for p in procs: p.start()
    for p in procs: p.join()
    print(counter.value)

In [ ]:
!python sol4.py

**Solution 5**

In [ ]:
%%writefile sol5.py
import multiprocessing as mp

def chunk_sum(numbers):
    return sum(numbers)

if __name__ == "__main__":
    all_numbers = list(range(1, 101))
    chunks = [all_numbers[i::4] for i in range(4)]    # four interleaved chunks
    with mp.Pool(4) as pool:
        partials = pool.map(chunk_sum, chunks)
    print("partials:", partials, "| total:", sum(partials))

In [ ]:
!python sol5.py

---

## Summary and Key Points

- Multiprocessing runs work in separate processes with independent interpreters, giving true parallelism for CPU-bound tasks, unlike threads which are limited by the Global Interpreter Lock.
- Code that starts processes must be guarded by `if __name__ == "__main__":`, especially because the `spawn` start method (default on macOS and Windows) re-imports the module.
- A `Pool` distributes the same function over many inputs with `pool.map`, returning results in input order, and is easier than managing processes by hand.
- Processes do not share memory: pass data with a `multiprocessing.Queue`, and share state through `Value` or `Array`, guarding updates with their built-in lock.
- Everything crossing a process boundary must be picklable (no lambdas or local functions); process and data-transfer overhead means multiprocessing suits substantial work, not tiny tasks.

### Next module: 5.3 — Executors (concurrent.futures)

The next module covers a higher-level interface to both threads and processes through `concurrent.futures`, with thread and process pool executors and `Future` objects.